# PI-Link QML: CodeBERT features + real IBM Quantum hardware
Runtime → Change runtime type → **T4 GPU**. Upload `colab_bundle.zip` when prompted, then run all cells top to bottom.

Outputs: `codebert_results.zip` (Part A) and `rq3_hardware.zip` (Part B). Attach both back in the chat.

In [ ]:
from google.colab import files
up = files.upload()  # choose colab_bundle.zip
!unzip -oq colab_bundle.zip && ls

In [ ]:
!pip -q install qiskit==2.* qiskit-aer qiskit-machine-learning qiskit-ibm-runtime xgboost transformers

## Part A: CodeBERT embeddings (GPU, ~10 min)

In [ ]:
!python embed_codebert.py --data pilink_pairs.csv --out codebert_emb.npz

Same protocol as the TF-IDF run (7 seeds, d = 4/8/12, repo-level split). Skips VQC and noise to keep it to about 1 hour on Colab CPU.

In [ ]:
!python -u qml_pilink.py --data pilink_pairs.csv --emb codebert_emb.npz --out results_codebert --seeds 7 --dims 4 8 12 --n_train 800 --n_test 400 --n_rank 200 | tee codebert.log | tail -3
!zip -q codebert_results.zip -r results_codebert codebert.log
files.download('codebert_results.zip')

## Part B: real IBM Quantum QPU (4 qubits)
Get a free API key at **quantum.cloud.ibm.com** (open plan = 10 QPU-minutes/month). The defaults (30+30 pairs, 512 shots, Z and ZZ1 maps, 4 jobs) are sized to fit inside that, roughly 4–7 QPU-minutes. Queue waits on the open plan can take a while; the cell blocks until jobs finish.

In [ ]:
import getpass
TOKEN = getpass.getpass('IBM Quantum API key: ')
INSTANCE = 'PASTE_YOUR_INSTANCE_CRN_HERE'

In [ ]:
args = f'--token {TOKEN}' + (f' --instance {INSTANCE}' if INSTANCE else '')
!python -u ibm_hardware.py $args --n 30 --shots 512 --out rq3_hardware
!zip -q rq3_hardware.zip -r rq3_hardware
files.download('rq3_hardware.zip')